In [6]:
from pathlib import Path
from dotenv import load_dotenv


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists() or (candidate / "pyproject.toml").exists():
            return candidate
    return start


project_root = find_project_root(Path.cwd())
env_path = project_root / ".env"
loaded = load_dotenv(env_path, override=False)
print("Załadowano .env:", loaded, "| ścieżka:", env_path)

Załadowano .env: True | ścieżka: C:\Users\Łukasz\PycharmProjects\air-quality-ml\.env


In [7]:
import pandas as pd
from sqlalchemy import text
from air_quality_ml.database import engine

In [5]:
import pandas as pd
from sqlalchemy import text
from air_quality_ml.database import engine

# Szybki test połączenia i liczników rekordów
counts_sql = text("""
    SELECT
        (SELECT COUNT(*) FROM stations) AS stations_cnt,
        (SELECT COUNT(*) FROM sensors) AS sensors_cnt,
        (SELECT COUNT(*) FROM measurements) AS measurements_cnt,
        (SELECT COUNT(*) FROM meteo_measurements) AS meteo_cnt;
""")

with engine.connect() as conn:
    print(pd.read_sql(counts_sql, conn))

   stations_cnt  sensors_cnt  measurements_cnt  meteo_cnt
0            78          414           9114945    3288024


In [8]:
station = text("""
    select * from stations
""")

meteo_measurements = text("""
    select * from meteo_measurements
""")

sensors = text("""
    select * from sensors
""")

measurements = text("""
    select * from measurements
""")
with engine.connect() as conn:
    df_station = pd.read_sql(station, conn)
with engine.connect() as conn:
    df_meteo_measurements = pd.read_sql(meteo_measurements, conn)
with engine.connect() as conn:
    df_sensors = pd.read_sql(sensors, conn)
with engine.connect() as conn:
    df_measurements = pd.read_sql(measurements, conn)

In [9]:
from air_quality_ml.ml.data_checks import (
    audit_summary_df,
    check_meteo_columns_nulls,
    run_data_quality_audit,
)

gios_report = run_data_quality_audit(
    df=df_measurements,
    dataset_name="gios_measurements",
    group_col="sensor_id",
    key_cols=["sensor_id", "datetime"],
    value_col="value",
    min_allowed=0.0,
    max_allowed=2000.0,
    flatline_min_streak=6,
)

meteo_report = run_data_quality_audit(
    df=df_meteo_measurements,
    dataset_name="open_meteo",
    group_col="station_id",
    key_cols=["station_id", "datetime"],
    value_col="temperature_2m",
    min_allowed=-45.0,
    max_allowed=45.0,
    flatline_min_streak=12,
    extra_checks=[check_meteo_columns_nulls],
)

display(audit_summary_df([gios_report, meteo_report]).T)

,0,1
dataset,gios_measurements,open_meteo
duplikaty_i_nulle.total_rows,9115411,3288024
duplikaty_i_nulle.duplicate_keys,0,0
duplikaty_i_nulle.null_values,0,0
duplikaty_i_nulle.null_pct,0.0,0.0
siatka_czasowa.groups_cnt,414,78
siatka_czasowa.min_dt,2021-11-01 00:00:00,2021-11-01 00:00:00
siatka_czasowa.max_dt,2026-11-22 16:00:00,2026-11-22 23:00:00
siatka_czasowa.missing_hours_total,7230692,171432
siatka_czasowa.future_rows,0,390
